# 110 · Long flux pulse on one SHFQC

**Level:** basic · **Topics:** Gen2, SHFQC, flux, calibration, pulses, markers, sweeps · **Type:** write the code

Everything runs on a **single SHFQC+**: drive, readout and also the flux line. There is no HDAWG.

The sequence, repeated for every sweep point:

| Time | `flux` | `drive` |
| --- | --- | --- |
| 0 | start of a 50 µs square pulse | 100 ns marker, no pulse |
| 49.8 µs | still on | first pulse, **amplitude** swept |
| 49.9 µs | still on | second pulse, **phase** swept |
| 50 µs | end | readout starts |

Amplitude and phase are swept **together** (one sweep, 21 points).

1. Calibrate the flux line so that an SHFQC signal generator channel can play it.
2. Write the sequence.
3. Make the 50 µs pulse fit in the memory of the instrument.

Cells with a `TODO` are yours to complete. Everything runs in emulation, no instrument needed.

In [ ]:
import numpy as np

from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.simple import *

## Device setup

One SHFQC+. The flux line uses the second signal generator channel.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/flux", ports="SGCHANNELS/1/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Step 1: calibration

The drive is a microwave signal at 5.1 GHz. The flux pulse is a baseband (DC) pulse:
the signal generator channel must use its **LF path**, with a centre frequency of 0 Hz.

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Calibration properties](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/02_calibration_properties.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
# TODO: complete the calibration of the flux line
calibration["q0/flux"] = SignalCalibration(
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Step 2: the sequence

📖 [Pulse commands](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/00_pulse_commands.html) ·
[Triggers and markers](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/07_trigger_marker.html) ·
[Amplitude and phase of pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/06_pulses_amplitude_and_phase.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html)

In [ ]:
flux_pulse = pulse_library.const(length=50e-6, amplitude=0.5)
drive_pulse = pulse_library.gaussian(length=100e-9, amplitude=1.0)
readout_pulse = pulse_library.const(length=1e-6, amplitude=0.5)

flux_length = 50e-6
marker_length = 100e-9

amplitudes = LinearSweepParameter(start=0.1, stop=1.0, count=21)
phases = LinearSweepParameter(start=0, stop=2 * np.pi, count=21)

In [ ]:
@dsl.experiment(signals=["drive", "flux", "measure", "acquire"])
def flux_sequence():
    with dsl.acquire_loop_rt(count=16):
        # TODO: sweep `amplitudes` and `phases` together, in one sweep
        with dsl.section(name="flux") as flux_section:
            dsl.play("flux", flux_pulse)
            # TODO: on "drive", play a marker of `marker_length` without a pulse
            # TODO: wait until 200 ns before the end of the flux pulse
            # TODO: play `drive_pulse` twice: first with the swept amplitude,
            #       then with the swept phase
        with dsl.section(name="readout", play_after=flux_section):
            dsl.measure(
                measure_signal="measure",
                acquire_signal="acquire",
                measure_pulse=readout_pulse,
                integration_kernel=readout_pulse,
                handle="result",
                reset_delay=1e-6,
            )


experiment = flux_sequence()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "flux": "q0/flux",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Step 3: compile and run

Once steps 1 and 2 are complete, the compilation fails with `resource limitation exceeded`.
The flux pulse must stay 50 µs long. Change **one argument** to make it fit.

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)

**Expected result:** `(21,)`

Look at the end of the first shot. The two drive pulses must sit at the end of the flux pulse, and the readout must start when the flux pulse ends (50 µs).

In [ ]:
plot_simulation(
    compiled, start_time=49.5e-6, length=2e-6, signals=["flux", "drive", "measure"]
)

## Questions

1. What happens if the flux line keeps the default port mode and gets its own local oscillator frequency, for example 1 GHz?
2. Would chunking the sweep solve the memory problem?
3. The first drive pulse is swept in amplitude, the second in phase. Why do these 21 sweep points not fill the waveform memory?
4. Which pulses can be compressed?

<details>
<summary>Hint 1</summary>

`SignalCalibration` has a `port_mode`. In LF mode, `local_oscillator` sets the centre frequency of the channel.
</details>

<details>
<summary>Hint 2</summary>

A marker without a pulse needs to know when it starts and how long it is:
`dsl.play("drive", pulse=None, marker={"marker1": {"start": ..., "length": ...}})`
</details>

<details>
<summary>Hint 3</summary>

One sweep over two parameters: `with dsl.sweep(parameter=[a, b]) as (x, y):`
</details>

<details>
<summary>Hint 4</summary>

Look at the arguments of `pulse_library.const`.
</details>

The solution is in [solution.ipynb](solution.ipynb).